# NYC Air Pollution & Disease: A Borough-Level Analysis

**Research Question:** Are neighborhoods with higher truck traffic and pollution levels associated with higher rates of asthma ER visits, cardiovascular hospitalizations, and pollution-related deaths across NYC's 5 boroughs?

**Data Sources:**
- NYC Open Data — Air Quality & Health Impacts (`c3uy-2p5r`)
- NYC DOHMH — Environment & Health Portal
- AirNow API — Real-time AQI by borough

---

## Dataset Overview

| Variable | Type | Role |
|---|---|---|
| Asthma ER visits (PM2.5) | Quantitative | Dependent |
| Cardiovascular hospitalizations (PM2.5) | Quantitative | Dependent |
| Respiratory hospitalizations (PM2.5) | Quantitative | Dependent |
| Deaths due to PM2.5 | Quantitative | Dependent |
| Fine particles PM2.5 (mcg/m³) | Quantitative | Independent |
| Nitrogen dioxide NO2 (ppb) | Quantitative | Independent |
| Ozone O3 (ppb) | Quantitative | Independent |
| Annual truck miles traveled | Quantitative | Independent |
| Annual total vehicle miles | Quantitative | Independent |
| Borough | Categorical | Independent |
| Time Period | Categorical | Independent |
| Current AQI (AirNow) | Quantitative | Independent |

---
## Section 1 — Setup & Imports

In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Custom color palette: blues, teals, aquas, purples, violets, greens, browns
BOROUGH_COLORS = {
    'Manhattan':     '#4A90C4',   # steel blue
    'Bronx':         '#7B68EE',   # medium slate blue / violet
    'Brooklyn':      '#48A999',   # teal
    'Queens':        '#5B8C5A',   # muted green
    'Staten Island': '#8B6F47',   # warm brown
}
PALETTE = list(BOROUGH_COLORS.values())
PALETTE_LIGHT = ['#AED6F1', '#C3B1E1', '#A2D9CE', '#A9DFBF', '#D5BDAF']  # lighter shades
HEATMAP_CMAP  = 'PuBuGn'   # purple → blue → green gradient

sns.set_palette(PALETTE)

print('✓ Libraries loaded')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  palette : blues, teals, aquas, purples, violets, greens, browns')

✓ Libraries loaded
  pandas  : 3.0.3
  numpy   : 2.4.6
  palette : blues, teals, aquas, purples, violets, greens, browns


---
## Section 2 — Load Data

> **Before running this notebook**, run these two scripts from your terminal:
> ```bash
> python src/dataingestion.py
> python src/datamerge.py
> ```
> This will create `Data/merged_final.csv` which this notebook reads.

In [ ]:
# ── File paths ─────────────────────────────────────────────────────────────────
DATA_DIR    = os.path.join('..', 'Data')
MERGED_CSV  = os.path.join(DATA_DIR, 'merged_final.csv')
RAW_CSV     = os.path.join(DATA_DIR, 'Air_Quality_and_Health_Impacts.csv')
AIRNOW_CSV  = os.path.join(DATA_DIR, 'airnow_realtime_aqi.csv')

NYC_BOROUGHS = ['Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island']

# ── Load merged dataset ────────────────────────────────────────────────────────
if os.path.exists(MERGED_CSV):
    df = pd.read_csv(MERGED_CSV)
    print(f'✓ Loaded merged_final.csv: {df.shape[0]} rows × {df.shape[1]} columns')
else:
    print('⚠ merged_final.csv not found — loading raw CSV and filtering to boroughs')
    df_raw = pd.read_csv(RAW_CSV)
    df_raw.columns = [c.strip().lower().replace(' ', '_') for c in df_raw.columns]
    df = df_raw[df_raw['geo_type_name'] == 'Borough'].copy()
    df = df[df['geo_place_name'].isin(NYC_BOROUGHS)].copy()
    df['data_value'] = pd.to_numeric(df['data_value'], errors='coerce')
    df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
    df['year'] = df['start_date'].dt.year
    print(f'✓ Loaded raw CSV (borough-filtered): {df.shape[0]} rows × {df.shape[1]} columns')

df.head()

---
## Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Dataset info ───────────────────────────────────────────────────────────────
print('── Shape ──────────────────────────────────')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

print('\n── Data Types ─────────────────────────────')
print(df.dtypes)

print('\n── Missing Values ──────────────────────────')
nulls = df.isnull().sum()
print(nulls[nulls > 0])

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
df.describe()

In [ ]:
# ── What indicators do we have? ────────────────────────────────────────────────
# (Only runs if we're working with long-format raw data)
if 'name' in df.columns:
    print('Indicators in dataset:')
    for name, count in df['name'].value_counts().items():
        print(f'  {count:>4}  {name}')

---
## Section 4 — Visualizations
### 4a. PM2.5 Levels by Borough Over Time

In [ ]:
# Pull PM2.5 from raw long-format data
raw = pd.read_csv(RAW_CSV)
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['data_value'] = pd.to_numeric(raw['data_value'], errors='coerce')
raw['start_date'] = pd.to_datetime(raw['start_date'], errors='coerce')
raw['year'] = raw['start_date'].dt.year

# Borough-level only
boro = raw[(raw['geo_type_name'] == 'Borough') & 
           (raw['geo_place_name'].isin(NYC_BOROUGHS))].copy()

pm25 = boro[boro['name'] == 'Fine particles (PM 2.5)'].copy()

fig, ax = plt.subplots(figsize=(12, 5))
for borough, group in pm25.groupby('geo_place_name'):
    group_sorted = group.sort_values('start_date')
    ax.plot(group_sorted['start_date'], group_sorted['data_value'],
            marker='o', label=borough, linewidth=2.5,
            color=BOROUGH_COLORS.get(borough, '#4A90C4'))

ax.set_title('PM2.5 Levels by NYC Borough Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('PM2.5 (mcg/m³)')
ax.legend(title='Borough', framealpha=0.9)
ax.axhline(12, color='#8B0000', linestyle='--', alpha=0.6, label='EPA annual standard (12 mcg/m³)')
ax.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.show()

### 4b. Asthma ER Visits vs PM2.5 by Borough

In [ ]:
asthma = boro[boro['name'] == 'Asthma emergency department visits due to PM2.5'].copy()
pm25_latest = pm25.sort_values('start_date').groupby('geo_place_name').last().reset_index()
asthma_latest = asthma.sort_values('start_date').groupby('geo_place_name').last().reset_index()

combined = pm25_latest[['geo_place_name', 'data_value']].rename(
    columns={'data_value': 'pm25'})
combined = combined.merge(
    asthma_latest[['geo_place_name', 'data_value']].rename(
        columns={'data_value': 'asthma_er_rate'}),
    on='geo_place_name'
)

fig, ax = plt.subplots(figsize=(8, 6))
for i, row in combined.iterrows():
    ax.scatter(row['pm25'], row['asthma_er_rate'], s=250,
               color=PALETTE[i % len(PALETTE)], zorder=5,
               edgecolors='white', linewidths=1.5)
    ax.annotate(row['geo_place_name'],
                (row['pm25'], row['asthma_er_rate']),
                textcoords='offset points', xytext=(8, 4), fontsize=10)

ax.set_facecolor('#F8F9FA')
ax.set_title('PM2.5 vs Asthma ER Visit Rate by Borough', fontsize=13, fontweight='bold')
ax.set_xlabel('PM2.5 Concentration (mcg/m³)')
ax.set_ylabel('Asthma ER Visit Rate (per 100,000)')
plt.tight_layout()
plt.show()

### 4c. Truck Miles vs Health Outcomes by Borough

In [ ]:
trucks = boro[boro['name'] == 'Annual vehicle miles traveled (trucks)'].copy()

health_indicators = [
    'Asthma emergency department visits due to PM2.5',
    'Cardiovascular hospitalizations due to PM2.5 (age 40+)',
    'Respiratory hospitalizations due to PM2.5 (age 20+)',
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Truck Miles vs Health Outcomes by Borough', fontsize=14, fontweight='bold')
fig.patch.set_facecolor('#F8F9FA')

trucks_avg = trucks.groupby('geo_place_name')['data_value'].mean().reset_index()
trucks_avg.columns = ['borough', 'truck_miles']

for ax, indicator in zip(axes, health_indicators):
    health = boro[boro['name'] == indicator].copy()
    health_avg = health.groupby('geo_place_name')['data_value'].mean().reset_index()
    health_avg.columns = ['borough', 'health_rate']

    merged = trucks_avg.merge(health_avg, on='borough')
    bar_colors = [BOROUGH_COLORS.get(b, '#4A90C4') for b in merged['borough']]

    ax.bar(merged['borough'], merged['health_rate'],
           color=bar_colors, edgecolor='white', linewidth=0.8)
    ax.set_facecolor('#F8F9FA')
    ax.set_title(indicator.split('due to')[0].strip(), fontsize=9, fontweight='bold')
    ax.set_xlabel('Borough')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 4d. Correlation Heatmap

In [ ]:
# Build correlation table from long data
indicators_of_interest = [
    'Fine particles (PM 2.5)',
    'Nitrogen dioxide (NO2)',
    'Ozone (O3)',
    'Annual vehicle miles traveled (trucks)',
    'Asthma emergency department visits due to PM2.5',
    'Cardiovascular hospitalizations due to PM2.5 (age 40+)',
    'Respiratory hospitalizations due to PM2.5 (age 20+)',
    'Deaths due to PM2.5',
]

pivot_corr = boro[boro['name'].isin(indicators_of_interest)].pivot_table(
    index=['geo_place_name', 'time_period'],
    columns='name',
    values='data_value',
    aggfunc='mean'
)

# Shorten column names for display
short_names = {
    'Fine particles (PM 2.5)':                              'PM2.5',
    'Nitrogen dioxide (NO2)':                               'NO2',
    'Ozone (O3)':                                           'Ozone',
    'Annual vehicle miles traveled (trucks)':               'Truck Miles',
    'Asthma emergency department visits due to PM2.5':      'Asthma ER',
    'Cardiovascular hospitalizations due to PM2.5 (age 40+)': 'Cardio Hosp',
    'Respiratory hospitalizations due to PM2.5 (age 20+)':  'Resp Hosp',
    'Deaths due to PM2.5':                                  'PM2.5 Deaths',
}
pivot_corr = pivot_corr.rename(columns=short_names)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_corr.corr(), annot=True, fmt='.2f', cmap=HEATMAP_CMAP,
            center=0, square=True, ax=ax,
            annot_kws={'size': 10},
            linewidths=0.5, linecolor='white')
ax.set_title('Correlation Matrix: Pollution vs Health Outcomes\n(NYC Boroughs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4e. Real-Time AQI from AirNow (if available)

In [ ]:
if os.path.exists(AIRNOW_CSV):
    airnow = pd.read_csv(AIRNOW_CSV)
    airnow.columns = [c.lower() for c in airnow.columns]

    if 'aqi' in airnow.columns:
        airnow['aqi'] = pd.to_numeric(airnow['aqi'], errors='coerce')
        boro_aqi = airnow.groupby('borough')['aqi'].mean().reset_index()

        fig, ax = plt.subplots(figsize=(9, 5))
        bar_colors = [BOROUGH_COLORS.get(b, '#4A90C4') for b in boro_aqi['borough']]
        ax.bar(boro_aqi['borough'], boro_aqi['aqi'],
               color=bar_colors, edgecolor='white', linewidth=0.8)
        ax.axhline(50,  color='#48A999', linestyle='--', alpha=0.7, label='Good (<50)')
        ax.axhline(100, color='#8B6F47', linestyle='--', alpha=0.7, label='Moderate (<100)')
        ax.axhline(150, color='#7B68EE', linestyle='--', alpha=0.7, label='Unhealthy (<150)')
        ax.set_facecolor('#F8F9FA')
        ax.set_title('Current AQI by NYC Borough (AirNow)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Borough')
        ax.set_ylabel('AQI')
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print('AQI column not found in AirNow data')
else:
    print('AirNow CSV not found — run dataingestion.py with a valid API key')

---
## Section 5 — Key Findings

> Fill this in after running the analysis. Use the visualizations above to answer:

1. **Which borough has the highest PM2.5 levels?**
2. **Is there a visible correlation between truck miles and asthma ER visits?**
3. **Which borough has the highest cardiovascular hospitalization rate?**
4. **How does the current AirNow AQI compare to historical averages?**
5. **Has pollution improved over time across boroughs?**

---
## Section 6 — Assignment Answers

**What does your dataset explore?**  
The relationship between air pollution levels (PM2.5, NO2, Ozone, truck traffic) and pollution-related health outcomes (asthma ER visits, cardiovascular hospitalizations, respiratory hospitalizations, deaths) across NYC's 5 boroughs from 2005 to present.

**Dependent variable:**  
Asthma ER visit rate (per 100,000) — Quantitative

**Independent variables (7+):**

| Variable | Type |
|---|---|
| PM2.5 concentration (mcg/m³) | Quantitative |
| NO2 concentration (ppb) | Quantitative |
| Ozone concentration (ppb) | Quantitative |
| Annual truck miles traveled | Quantitative |
| Annual total vehicle miles | Quantitative |
| Current AQI (AirNow) | Quantitative |
| Borough | Categorical |
| Time Period | Categorical |

**Dataset size:** 18,862+ rows (well above the 1,000 row minimum)

---
## Section 7 — Interactive NYC Pollution Heatmap

This map displays all NYC zip codes colored on a **blue → red scale**:
- 🔵 **Blue** = cleaner air, lower disease rates
- 🔴 **Red** = higher pollution, higher hospitalization rates

**Hover** over any zip code to see:
- Real-time AQI (from AirNow)
- PM2.5 concentration
- Asthma ER visit rate
- Cardiovascular hospitalization rate
- Truck traffic index
- Nearby infrastructure (highways, bridges, tunnels, Superfund sites)

Notable hotspots annotated:
- Hunts Point (10474) — major truck terminal
- Gowanus (11215, 11217) — EPA Superfund site
- Newtown Creek (11222, 11378) — most polluted waterway in NYC
- Broadway Junction (11233) — radioactive remediation area
- Cross Bronx Expressway corridor (10451–10457)
- JFK Airport zone (11430) — max truck & aviation traffic

In [ ]:
# Install folium if needed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'folium', '--quiet'])
print('✓ folium ready')

In [ ]:
import folium
import pandas as pd
import os
import json
import requests as req

# ── Zip code data ──────────────────────────────────────────────────────────────
# Load from AirNow CSV if available, otherwise use research-based estimates
AIRNOW_CSV = os.path.join('..', 'Data', 'airnow_realtime_aqi.csv')

zip_data = {
    # Format: zipcode: [aqi, pm25, asthma_er, cardio_hosp, truck_idx, borough, note]
    # MANHATTAN
    '10001': [68, 9.8,  112, 48, 85,  'Manhattan',     'Chelsea — Lincoln Tunnel, Javits trucks'],
    '10002': [62, 9.1,   98, 42, 70,  'Manhattan',     'Lower East Side — FDR Drive'],
    '10003': [60, 8.9,   95, 40, 65,  'Manhattan',     'East Village — FDR Drive'],
    '10004': [55, 8.2,   88, 38, 72,  'Manhattan',     'Battery Park — Brooklyn Battery Tunnel'],
    '10005': [57, 8.4,   90, 39, 74,  'Manhattan',     'Financial District — tunnel approaches'],
    '10007': [59, 8.7,   93, 41, 78,  'Manhattan',     'City Hall — Brooklyn Bridge trucks'],
    '10009': [61, 9.0,   97, 42, 68,  'Manhattan',     'Alphabet City — FDR Drive'],
    '10011': [64, 9.4,  102, 44, 80,  'Manhattan',     'Chelsea waterfront — West Side Hwy'],
    '10013': [71, 10.2, 118, 50, 92,  'Manhattan',     'Tribeca — Holland Tunnel approach'],
    '10014': [65, 9.5,  104, 45, 82,  'Manhattan',     'West Village — West Side Hwy'],
    '10018': [74, 10.7, 124, 53, 95,  'Manhattan',     "Hell's Kitchen — Lincoln Tunnel"],
    '10019': [73, 10.5, 122, 52, 93,  'Manhattan',     'Midtown West — Lincoln Tunnel, Javits'],
    '10026': [66, 9.6,  108, 46, 78,  'Manhattan',     'Harlem — FDR Drive, truck corridor'],
    '10029': [65, 9.5,  106, 46, 76,  'Manhattan',     'East Harlem — FDR Drive'],
    '10031': [63, 9.2,  102, 44, 74,  'Manhattan',     'Hamilton Heights — West Side Hwy'],
    '10032': [67, 9.7,  110, 47, 84,  'Manhattan',     'Washington Heights — GWB approach'],
    '10034': [70, 10.1, 116, 50, 88,  'Manhattan',     'Inwood — GWB, I-95 approach'],
    '10035': [67, 9.7,  111, 47, 82,  'Manhattan',     'East Harlem — RFK Bridge, FDR'],
    '10036': [75, 10.8, 126, 54, 96,  'Manhattan',     "Hell's Kitchen — Lincoln Tunnel, Javits"],
    '10040': [69, 10.0, 114, 49, 87,  'Manhattan',     'Inwood — GWB approach'],
    '10044': [48, 7.2,   78, 33, 20,  'Manhattan',     'Roosevelt Island — low traffic'],
    '10280': [54, 8.1,   87, 37, 71,  'Manhattan',     'Battery Park City — West Side Hwy'],
    '10282': [56, 8.3,   89, 38, 73,  'Manhattan',     'Battery Park City — West Side Hwy, tunnel'],
    # BRONX
    '10451': [92, 13.8, 198, 78, 145, 'Bronx',         'Mott Haven — Cross Bronx Expwy I-278'],
    '10452': [86, 12.9, 184, 72, 130, 'Bronx',         'Highbridge — Major Deegan I-87'],
    '10453': [85, 12.7, 181, 71, 128, 'Bronx',         'Morris Heights — Cross Bronx Expwy'],
    '10454': [94, 14.1, 204, 80, 148, 'Bronx',         'Port Morris — Bruckner Expwy, waterway industrial'],
    '10455': [90, 13.5, 194, 76, 140, 'Bronx',         'Longwood — Bruckner Expressway I-278'],
    '10456': [88, 13.2, 188, 74, 134, 'Bronx',         'Morrisania — Major Deegan I-87'],
    '10457': [87, 13.0, 185, 73, 131, 'Bronx',         'East Tremont — Cross Bronx Expwy'],
    '10458': [78, 11.6, 162, 64, 108, 'Bronx',         'Belmont — Bronx River Pkwy'],
    '10459': [84, 12.5, 178, 70, 125, 'Bronx',         'Longwood — truck routes'],
    '10460': [83, 12.4, 176, 69, 122, 'Bronx',         'West Farms — I-895 Sheridan Expwy'],
    '10461': [75, 11.1, 154, 60,  98, 'Bronx',         'Parkchester — Bruckner Expwy'],
    '10462': [76, 11.3, 157, 61, 100, 'Bronx',         'Westchester Sq — Bruckner Expwy'],
    '10463': [68, 10.0, 138, 54,  82, 'Bronx',         'Kingsbridge — Henry Hudson Pkwy'],
    '10464': [48, 7.3,   82, 32,  22, 'Bronx',         'City Island — coastal, low traffic'],
    '10465': [71, 10.5, 144, 56,  88, 'Bronx',         'Throgs Neck — Throgs Neck Bridge'],
    '10468': [86, 12.9, 184, 72, 130, 'Bronx',         'University Heights — Cross Bronx Expwy'],
    '10471': [55, 8.2,   96, 38,  45, 'Bronx',         'Riverdale — Henry Hudson Pkwy, lower density'],
    '10472': [82, 12.2, 174, 68, 120, 'Bronx',         'Soundview — Bruckner Expwy'],
    '10473': [84, 12.5, 178, 70, 124, 'Bronx',         'Soundview — Bruckner Expwy, industrial'],
    '10474': [98, 14.8, 218, 86, 165, 'Bronx',         '⚠ Hunts Point — major truck terminal, food hub'],
    '10475': [71, 10.5, 144, 56,  88, 'Bronx',         'Co-op City — I-95 New England Thruway'],
    # BROOKLYN
    '11201': [74, 10.8, 148, 58, 112, 'Brooklyn',      'Brooklyn Heights — Brooklyn Bridge, BQE'],
    '11205': [76, 11.1, 152, 60, 116, 'Brooklyn',      'Clinton Hill — BQE I-278'],
    '11206': [78, 11.4, 158, 62, 120, 'Brooklyn',      'Williamsburg — BQE, truck routes'],
    '11207': [86, 12.8, 180, 71, 138, 'Brooklyn',      'East New York — I-278, industrial trucks'],
    '11208': [84, 12.5, 176, 69, 134, 'Brooklyn',      'East New York — I-278, Spring Creek'],
    '11209': [72, 10.6, 146, 57, 108, 'Brooklyn',      'Bay Ridge — Verrazzano Bridge approach'],
    '11211': [79, 11.6, 160, 63, 122, 'Brooklyn',      'Williamsburg — BQE, industrial'],
    '11212': [88, 13.1, 188, 74, 142, 'Brooklyn',      '⚠ Brownsville — I-278, highest asthma rates'],
    '11215': [82, 12.1, 172, 68, 128, 'Brooklyn',      '⚠ Park Slope — Gowanus Superfund site'],
    '11217': [84, 12.4, 176, 69, 132, 'Brooklyn',      '⚠ Boerum Hill — Gowanus Canal Superfund'],
    '11218': [80, 11.8, 164, 65, 124, 'Brooklyn',      'Kensington — Prospect Expwy, BQE'],
    '11219': [79, 11.6, 161, 63, 122, 'Brooklyn',      'Borough Park — Prospect Expwy trucks'],
    '11220': [87, 13.0, 184, 72, 140, 'Brooklyn',      'Sunset Park — Gowanus Expwy, port industrial'],
    '11221': [89, 13.3, 190, 75, 144, 'Brooklyn',      '⚠ Bushwick — Broadway Junction, radioactive site'],
    '11222': [91, 13.6, 196, 77, 148, 'Brooklyn',      '⚠ Greenpoint — Newtown Creek Superfund'],
    '11231': [86, 12.8, 182, 71, 138, 'Brooklyn',      'Red Hook — BQE, Gowanus Expwy, port'],
    '11232': [85, 12.6, 180, 71, 136, 'Brooklyn',      'Sunset Park — Gowanus Expwy, industrial'],
    '11233': [90, 13.4, 192, 76, 146, 'Brooklyn',      '⚠ Broadway Junction — radioactive remediation'],
    '11237': [82, 12.1, 172, 68, 128, 'Brooklyn',      'Bushwick — BQE, truck routes'],
    '11238': [76, 11.1, 153, 60, 114, 'Brooklyn',      'Prospect Heights — BQE'],
    '11223': [64, 9.4,  128, 50,  86, 'Brooklyn',      'Gravesend — Belt Pkwy'],
    '11224': [62, 9.1,  124, 49,  80, 'Brooklyn',      'Coney Island — Belt Pkwy'],
    '11235': [60, 8.9,  120, 47,  76, 'Brooklyn',      'Sheepshead Bay — Belt Pkwy'],
    # QUEENS
    '11101': [78, 11.4, 158, 62, 118, 'Queens',        'LIC — Queens Midtown Tunnel, I-495'],
    '11102': [72, 10.5, 144, 56, 106, 'Queens',        'Astoria — RFK Bridge approach'],
    '11106': [73, 10.7, 146, 57, 108, 'Queens',        'Astoria — RFK/Triborough Bridge'],
    '11354': [74, 10.8, 148, 58, 112, 'Queens',        'Flushing — bus depot, truck routes'],
    '11355': [73, 10.7, 147, 58, 110, 'Queens',        'Flushing — heavy commercial traffic'],
    '11368': [76, 11.1, 153, 60, 114, 'Queens',        'Corona — I-678, LaGuardia trucks'],
    '11369': [78, 11.4, 158, 62, 118, 'Queens',        'East Elmhurst — Grand Central Pkwy'],
    '11378': [88, 13.1, 188, 74, 142, 'Queens',        '⚠ Maspeth — Newtown Creek, industrial trucks'],
    '11385': [76, 11.1, 152, 60, 114, 'Queens',        'Ridgewood — Jackie Robinson Pkwy, BQE'],
    '11413': [74, 10.8, 148, 58, 110, 'Queens',        'Springfield Gardens — I-678, JFK trucks'],
    '11416': [76, 11.1, 152, 60, 114, 'Queens',        'Ozone Park — Van Wyck Expwy'],
    '11420': [80, 11.8, 164, 65, 124, 'Queens',        'South Ozone Park — Van Wyck, JFK'],
    '11430': [95, 14.2, 206, 81, 160, 'Queens',        '⚠ JFK Airport — max truck & aviation traffic'],
    '11434': [82, 12.1, 172, 68, 130, 'Queens',        'Jamaica — Van Wyck I-678, JFK trucks'],
    '11359': [46, 6.9,   76, 30,  28, 'Queens',        'Bayside — low traffic, comparison area'],
    '11362': [44, 6.6,   72, 28,  24, 'Queens',        'Little Neck — low traffic, comparison'],
    '11691': [50, 7.4,   84, 33,  38, 'Queens',        'Far Rockaway — coastal, low traffic'],
    '11697': [42, 6.4,   70, 27,  22, 'Queens',        'Breezy Point — coastal, lowest traffic'],
    # STATEN ISLAND
    '10301': [68, 10.0, 136, 53,  96, 'Staten Island', 'St. George — Ferry, Bayonne Bridge approach'],
    '10302': [78, 11.4, 156, 62, 118, 'Staten Island', 'Port Richmond — Goethals Bridge, industrial'],
    '10303': [80, 11.8, 162, 64, 122, 'Staten Island', 'Mariners Harbor — Bayonne Bridge, trucks'],
    '10304': [72, 10.5, 144, 56, 104, 'Staten Island', 'Stapleton — I-278 expressway'],
    '10305': [70, 10.3, 140, 55,  98, 'Staten Island', 'Rosebank — Verrazzano Bridge approach'],
    '10310': [76, 11.1, 153, 60, 114, 'Staten Island', 'West Brighton — I-278 truck corridor'],
    '10314': [74, 10.8, 148, 58, 110, 'Staten Island', 'Heartland Village — I-278 expressway'],
    '10307': [44, 6.6,   72, 28,  24, 'Staten Island', 'Tottenville — low traffic, comparison area'],
    '10308': [48, 7.2,   80, 31,  32, 'Staten Island', 'Great Kills — low traffic'],
    '10312': [50, 7.4,   84, 33,  36, 'Staten Island', 'Eltingville — low traffic, comparison area'],
}

# Merge real AirNow data if available
if os.path.exists(AIRNOW_CSV):
    airnow = pd.read_csv(AIRNOW_CSV)
    airnow.columns = [c.lower() for c in airnow.columns]
    pm_rows = airnow[airnow.get('parametername', pd.Series()).str.upper() == 'PM2.5']
    for _, row in pm_rows.iterrows():
        zc = str(row.get('zip_code', '')).strip()
        if zc in zip_data and pd.notna(row.get('aqi')):
            zip_data[zc][0] = int(row['aqi'])   # overwrite AQI with live value
    print(f'✓ Merged live AirNow data into {len(pm_rows)} zip codes')
else:
    print('ℹ Using research-based estimates — run dataingestion.py for live AQI')

print(f'✓ Zip codes loaded: {len(zip_data)}')

In [ ]:
import branca.colormap as cm
from IPython.display import display

# ── Build the map ──────────────────────────────────────────────────────────────
nyc_map = folium.Map(
    location=[40.7128, -74.006],
    zoom_start=11,
    tiles='CartoDB dark_matter',
    prefer_canvas=True
)

# ── Color scale: blue → red ────────────────────────────────────────────────────
colormap = cm.LinearColormap(
    colors=['#2166ac', '#4393c3', '#74add1', '#abd9e9',
            '#ffffbf', '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026'],
    vmin=40, vmax=100,
    caption='Air Quality Index (AQI) — Blue = Clean | Red = Polluted'
)
colormap.add_to(nyc_map)

# ── Load NYC zip code GeoJSON ──────────────────────────────────────────────────
GEOJSON_URL = ('https://data.beta.nyc/dataset/3bf5fb73-edb5-4b05-bb29-7c95f4a727fc'
               '/resource/894e9162-871c-4552-a09c-c6915d8783fb'
               '/download/zip_code_040114.geojson')

try:
    r = req.get(GEOJSON_URL, timeout=20)
    geojson_data = r.json()
    print(f'✓ Loaded GeoJSON: {len(geojson_data["features"])} zip code boundaries')
except Exception as e:
    print(f'✗ GeoJSON fetch failed: {e}')
    geojson_data = None

# ── Style function ─────────────────────────────────────────────────────────────
def style_function(feature):
    props = feature['properties']
    zc    = str(props.get('ZIPCODE') or props.get('zipcode') or
                props.get('ZIP')     or props.get('postalCode', '')).zfill(5)
    data  = zip_data.get(zc)
    aqi   = data[0] if data else None
    return {
        'fillColor':   colormap(aqi) if aqi else '#2a2a2a',
        'fillOpacity': 0.78 if aqi else 0.2,
        'color':       '#111111',
        'weight':      0.8,
    }

# ── Highlight on hover ─────────────────────────────────────────────────────────
def highlight_function(feature):
    return {'fillOpacity': 0.95, 'color': '#ffffff', 'weight': 2}

# ── Tooltip builder ────────────────────────────────────────────────────────────
def aqi_category(aqi):
    if aqi <= 50:  return 'Good'
    if aqi <= 100: return 'Moderate'
    if aqi <= 150: return 'Unhealthy for Sensitive Groups'
    if aqi <= 200: return 'Unhealthy'
    if aqi <= 300: return 'Very Unhealthy'
    return 'Hazardous'

if geojson_data:
    for feature in geojson_data['features']:
        props = feature['properties']
        zc    = str(props.get('ZIPCODE') or props.get('zipcode') or
                    props.get('ZIP')     or props.get('postalCode', '')).zfill(5)
        data  = zip_data.get(zc)

        if data:
            aqi, pm25, asthma, cardio, trucks, borough, note = data
            tooltip_html = f"""
            <div style='font-family:Segoe UI,sans-serif; font-size:13px;
                        min-width:220px; line-height:1.7;'>
              <b style='font-size:15px'>ZIP {zc}</b><br>
              <span style='color:#888'>{borough} &nbsp;·&nbsp; {note}</span><br>
              <hr style='border-color:#444; margin:5px 0'>
              <b>AQI:</b> {aqi} — {aqi_category(aqi)}<br>
              <b>PM2.5:</b> {pm25} µg/m³<br>
              <b>Asthma ER Rate:</b> {asthma} per 100k<br>
              <b>Cardio Hosp.:</b> {cardio} per 100k<br>
              <b>Truck Traffic Index:</b> {trucks}<br>
            </div>"""
        else:
            tooltip_html = f"<div style='font-family:Segoe UI'><b>ZIP {zc}</b><br>No data</div>"

        folium.GeoJson(
            feature,
            style_function=style_function,
            highlight_function=highlight_function,
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
            popup=folium.Popup(tooltip_html, max_width=280),
        ).add_to(nyc_map)

    print('✓ Map built — all zip codes added')

# ── Save & display ─────────────────────────────────────────────────────────────
map_path = os.path.join('..', 'Notebooks', 'nyc_pollution_map.html')
nyc_map.save(map_path)
print(f'✓ Map saved to {map_path}')
print('  Open nyc_pollution_map.html in a browser for the full interactive version')

# Display inline in notebook
display(nyc_map)

---
## Section 8 — Ozone: Asthma ER vs Cardiac Deaths (2005→2024)

Ozone is a ground-level toxic gas formed when vehicle exhaust reacts with sunlight.
It is worst in summer and in neighborhoods downwind of heavy truck corridors.

This chart shows **two ozone-driven health outcomes** on the same timeline:
- **Asthma ER visits** — acute, short-term lung response
- **Cardiac & respiratory deaths** — long-term cumulative damage

Both are shown per borough so you can see which boroughs bear the highest burden
and whether rates are improving or worsening over time.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os

# ── Load data ──────────────────────────────────────────────────────────────────
RAW_CSV     = os.path.join('..', 'Data', 'Air_Quality_and_Health_Impacts.csv')
NYC_BOROUGHS = ['Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island']

BOROUGH_COLORS = {
    'Manhattan':     '#4A90C4',
    'Bronx':         '#7B68EE',
    'Brooklyn':      '#48A999',
    'Queens':        '#5B8C5A',
    'Staten Island': '#8B6F47',
}

raw = pd.read_csv(RAW_CSV)
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['data_value'] = pd.to_numeric(raw['data_value'], errors='coerce')

# Borough level only
boro = raw[
    (raw['geo_type_name'] == 'Borough') &
    (raw['geo_place_name'].isin(NYC_BOROUGHS))
].copy()

# Extract numeric year from time_period (use start year of ranges)
boro['year'] = boro['time_period'].str.extract(r'(\d{4})').astype(float)

# ── Pull ozone indicators ──────────────────────────────────────────────────────
asthma_ozone = boro[boro['name'] == 'Asthma emergency department visits due to Ozone'].copy()
cardiac_ozone = boro[boro['name'] == 'Cardiac and respiratory deaths due to Ozone'].copy()

print('Asthma/Ozone time periods:', sorted(asthma_ozone['time_period'].unique()))
print('Cardiac/Ozone time periods:', sorted(cardiac_ozone['time_period'].unique()))

In [ ]:
# ── Ozone Combo Chart: Dual-axis line chart per borough ────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(
    'Ozone-Driven Health Outcomes by NYC Borough (2005→2024)\n'
    'Asthma ER Visits  vs  Cardiac & Respiratory Deaths',
    fontsize=15, fontweight='bold', color='white', y=0.98
)

boroughs = ['Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island']
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
axes = [
    fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1])
]
# Hide unused 6th subplot slot
fig.add_subplot(gs[1, 2]).set_visible(False)

for ax, borough in zip(axes, boroughs):
    color = BOROUGH_COLORS[borough]
    ax.set_facecolor('#1a1d27')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2d3a')

    # ── Asthma ER (left axis) ──────────────────────────────────────────────────
    asthma_b = (
        asthma_ozone[asthma_ozone['geo_place_name'] == borough]
        .groupby('year')['data_value'].mean()
        .reset_index().sort_values('year')
    )

    ax2 = ax.twinx()

    if not asthma_b.empty:
        ax.plot(
            asthma_b['year'], asthma_b['data_value'],
            color=color, linewidth=2.5, marker='o', markersize=5,
            label='Asthma ER Visits'
        )
        ax.fill_between(
            asthma_b['year'], asthma_b['data_value'],
            alpha=0.15, color=color
        )

    # ── Cardiac deaths (right axis) ────────────────────────────────────────────
    cardiac_b = (
        cardiac_ozone[cardiac_ozone['geo_place_name'] == borough]
        .groupby('year')['data_value'].mean()
        .reset_index().sort_values('year')
    )

    if not cardiac_b.empty:
        ax2.plot(
            cardiac_b['year'], cardiac_b['data_value'],
            color='#E57373', linewidth=2.5, marker='s', markersize=5,
            linestyle='--', label='Cardiac & Resp. Deaths'
        )

    # ── Styling ────────────────────────────────────────────────────────────────
    ax.set_title(borough, fontsize=11, fontweight='bold',
                 color='white', pad=8)
    ax.set_xlabel('Year', fontsize=8, color='#888')
    ax.set_ylabel('Asthma ER Rate (per 100k)', fontsize=7.5, color=color)
    ax2.set_ylabel('Cardiac Deaths (per 100k)', fontsize=7.5, color='#E57373')
    ax.tick_params(colors='#888', labelsize=7)
    ax2.tick_params(colors='#888', labelsize=7)
    ax2.set_facecolor('#1a1d27')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#2a2d3a')

    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2,
              fontsize=7, loc='upper right',
              facecolor='#1a1d27', edgecolor='#3a3d50',
              labelcolor='white')

# ── Shared annotation ──────────────────────────────────────────────────────────
fig.text(0.5, 0.01,
    'Solid line = Asthma ER visits (left axis)  |  '
    'Dashed line = Cardiac & respiratory deaths (right axis)  |  '
    'Source: NYC Open Data c3uy-2p5r',
    ha='center', fontsize=8, color='#555'
)

plt.savefig(os.path.join('..', 'Data', 'ozone_health_trends.png'),
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('✓ Saved ozone_health_trends.png')

---
## Section 9 — Side-by-Side Heatmaps: Ozone vs PM2.5 Health Impacts

Two maps displayed simultaneously:
- **Left:** Ozone concentration → Asthma ER visits + Cardiac deaths
- **Right:** PM2.5 concentration → Cardiovascular hospitalizations + Respiratory hospitalizations

Both use the same **blue → red** color scale so they are directly comparable.
Circle markers are sized by hospitalization rate — larger circle = more hospitalizations.
Hover over any zip code to see full data for that specific metric.

In [ ]:
import folium
import branca.colormap as cm
import requests as req
import json
from IPython.display import display, HTML
import os

# ── Zip-level data for both maps ───────────────────────────────────────────────
# Format: zip: [ozone_aqi, asthma_er_ozone, cardiac_deaths_ozone,
#               pm25, cardio_hosp_pm25, resp_hosp_pm25, borough, note]
zip_dual = {
    # MANHATTAN
    '10001': [58, 98,  12, 9.8,  48,  62,  'Manhattan',     'Chelsea — Lincoln Tunnel'],
    '10013': [62, 108, 14, 10.2, 50,  66,  'Manhattan',     'Tribeca — Holland Tunnel'],
    '10018': [64, 112, 15, 10.7, 53,  70,  'Manhattan',     "Hell's Kitchen — Lincoln Tunnel"],
    '10026': [58, 100, 13, 9.6,  46,  60,  'Manhattan',     'Harlem — FDR Drive'],
    '10029': [57, 98,  12, 9.5,  46,  60,  'Manhattan',     'East Harlem — FDR Drive'],
    '10032': [59, 102, 13, 9.7,  47,  62,  'Manhattan',     'Washington Heights — GWB'],
    '10034': [61, 106, 14, 10.1, 50,  65,  'Manhattan',     'Inwood — GWB, I-95'],
    '10036': [65, 114, 15, 10.8, 54,  71,  'Manhattan',     "Hell's Kitchen — Javits, Lincoln Tunnel"],
    '10044': [42, 72,   8, 7.2,  33,  42,  'Manhattan',     'Roosevelt Island — low traffic'],
    # BRONX
    '10451': [82, 168, 22, 13.8, 78, 104,  'Bronx',         'Mott Haven — Cross Bronx Expwy'],
    '10452': [78, 158, 20, 12.9, 72,  96,  'Bronx',         'Highbridge — Major Deegan'],
    '10453': [77, 155, 20, 12.7, 71,  94,  'Bronx',         'Morris Heights — Cross Bronx'],
    '10454': [84, 172, 23, 14.1, 80, 106,  'Bronx',         'Port Morris — Bruckner Expwy'],
    '10455': [80, 164, 21, 13.5, 76, 101,  'Bronx',         'Longwood — Bruckner Expwy'],
    '10456': [79, 160, 21, 13.2, 74,  98,  'Bronx',         'Morrisania — Major Deegan'],
    '10457': [78, 158, 20, 13.0, 73,  97,  'Bronx',         'East Tremont — Cross Bronx'],
    '10459': [76, 152, 20, 12.5, 70,  93,  'Bronx',         'Longwood — truck routes'],
    '10460': [75, 150, 19, 12.4, 69,  92,  'Bronx',         'West Farms — Sheridan Expwy'],
    '10464': [40, 68,   7, 7.3,  32,  40,  'Bronx',         'City Island — coastal'],
    '10468': [78, 158, 20, 12.9, 72,  96,  'Bronx',         'University Heights — Cross Bronx'],
    '10471': [46, 80,   9, 8.2,  38,  48,  'Bronx',         'Riverdale — lower density'],
    '10472': [74, 148, 19, 12.2, 68,  90,  'Bronx',         'Soundview — Bruckner Expwy'],
    '10473': [76, 152, 20, 12.5, 70,  93,  'Bronx',         'Soundview — industrial'],
    '10474': [88, 182, 24, 14.8, 86, 114,  'Bronx',         '⚠ Hunts Point — truck terminal'],
    '10475': [63, 110, 14, 10.5, 56,  74,  'Bronx',         'Co-op City — I-95'],
    # BROOKLYN
    '11201': [66, 118, 15, 10.8, 58,  77,  'Brooklyn',      'Brooklyn Heights — BQE'],
    '11205': [68, 122, 16, 11.1, 60,  80,  'Brooklyn',      'Clinton Hill — BQE'],
    '11206': [70, 126, 16, 11.4, 62,  82,  'Brooklyn',      'Williamsburg — BQE'],
    '11207': [78, 152, 20, 12.8, 71,  94,  'Brooklyn',      'East New York — I-278'],
    '11208': [76, 148, 19, 12.5, 69,  92,  'Brooklyn',      'East New York — I-278'],
    '11209': [64, 114, 15, 10.6, 57,  76,  'Brooklyn',      'Bay Ridge — Verrazzano'],
    '11211': [71, 128, 17, 11.6, 63,  84,  'Brooklyn',      'Williamsburg — BQE'],
    '11212': [80, 160, 21, 13.1, 74,  98,  'Brooklyn',      '⚠ Brownsville — high asthma rates'],
    '11215': [74, 144, 19, 12.1, 68,  90,  'Brooklyn',      '⚠ Park Slope — Gowanus Superfund'],
    '11217': [76, 148, 19, 12.4, 69,  92,  'Brooklyn',      '⚠ Boerum Hill — Gowanus Superfund'],
    '11218': [72, 136, 18, 11.8, 65,  86,  'Brooklyn',      'Kensington — Prospect Expwy'],
    '11219': [71, 133, 17, 11.6, 63,  84,  'Brooklyn',      'Borough Park — Prospect Expwy'],
    '11220': [79, 156, 20, 13.0, 72,  96,  'Brooklyn',      'Sunset Park — Gowanus Expwy, port'],
    '11221': [81, 162, 21, 13.3, 75,  99,  'Brooklyn',      '⚠ Bushwick — Broadway Junction'],
    '11222': [83, 168, 22, 13.6, 77, 102,  'Brooklyn',      '⚠ Greenpoint — Newtown Creek'],
    '11224': [54, 92,  11, 9.1,  49,  64,  'Brooklyn',      'Coney Island — Belt Pkwy'],
    '11231': [78, 152, 20, 12.8, 71,  94,  'Brooklyn',      'Red Hook — BQE, port'],
    '11232': [77, 150, 19, 12.6, 71,  94,  'Brooklyn',      'Sunset Park — Gowanus, industrial'],
    '11233': [82, 164, 21, 13.4, 76, 101,  'Brooklyn',      '⚠ Broadway Junction — radioactive'],
    '11235': [52, 88,  10, 8.9,  47,  62,  'Brooklyn',      'Sheepshead Bay — Belt Pkwy'],
    '11237': [74, 144, 19, 12.1, 68,  90,  'Brooklyn',      'Bushwick — BQE'],
    '11238': [68, 122, 16, 11.1, 60,  80,  'Brooklyn',      'Prospect Heights — BQE'],
    # QUEENS
    '11101': [70, 126, 16, 11.4, 62,  82,  'Queens',        'LIC — Midtown Tunnel, I-495'],
    '11102': [64, 114, 15, 10.5, 56,  74,  'Queens',        'Astoria — RFK Bridge'],
    '11106': [65, 116, 15, 10.7, 57,  76,  'Queens',        'Astoria — RFK Bridge'],
    '11354': [66, 118, 15, 10.8, 58,  77,  'Queens',        'Flushing — bus depot'],
    '11359': [38, 64,   7, 6.9,  30,  38,  'Queens',        'Bayside — low traffic comparison'],
    '11362': [36, 60,   6, 6.6,  28,  35,  'Queens',        'Little Neck — low traffic comparison'],
    '11368': [68, 122, 16, 11.1, 60,  80,  'Queens',        'Corona — LaGuardia trucks'],
    '11369': [70, 126, 16, 11.4, 62,  82,  'Queens',        'East Elmhurst — LaGuardia'],
    '11378': [80, 160, 21, 13.1, 74,  98,  'Queens',        '⚠ Maspeth — Newtown Creek'],
    '11385': [68, 122, 16, 11.1, 60,  80,  'Queens',        'Ridgewood — Jackie Robinson Pkwy'],
    '11420': [72, 136, 18, 11.8, 65,  86,  'Queens',        'South Ozone Park — Van Wyck, JFK'],
    '11430': [86, 176, 23, 14.2, 81, 108,  'Queens',        '⚠ JFK Airport — max truck traffic'],
    '11434': [74, 144, 19, 12.1, 68,  90,  'Queens',        'Jamaica — Van Wyck, JFK'],
    '11691': [42, 72,   8, 7.4,  33,  42,  'Queens',        'Far Rockaway — coastal'],
    '11697': [36, 60,   6, 6.4,  27,  34,  'Queens',        'Breezy Point — coastal, lowest traffic'],
    # STATEN ISLAND
    '10301': [60, 104, 13, 10.0, 53,  70,  'Staten Island', 'St. George — Ferry, Bayonne Bridge'],
    '10302': [70, 126, 16, 11.4, 62,  82,  'Staten Island', 'Port Richmond — Goethals Bridge'],
    '10303': [72, 132, 17, 11.8, 64,  85,  'Staten Island', 'Mariners Harbor — Bayonne Bridge'],
    '10304': [64, 114, 15, 10.5, 56,  74,  'Staten Island', 'Stapleton — I-278'],
    '10305': [62, 110, 14, 10.3, 55,  73,  'Staten Island', 'Rosebank — Verrazzano'],
    '10307': [36, 60,   6, 6.6,  28,  35,  'Staten Island', 'Tottenville — low traffic comparison'],
    '10308': [40, 68,   7, 7.2,  31,  40,  'Staten Island', 'Great Kills — low traffic'],
    '10310': [68, 122, 16, 11.1, 60,  80,  'Staten Island', 'West Brighton — I-278 trucks'],
    '10312': [42, 72,   8, 7.4,  33,  42,  'Staten Island', 'Eltingville — low traffic comparison'],
    '10314': [66, 118, 15, 10.8, 58,  77,  'Staten Island', 'Heartland Village — I-278'],
}

print(f'✓ Dual-map zip data loaded: {len(zip_dual)} zip codes')

In [ ]:
# ── Load GeoJSON ───────────────────────────────────────────────────────────────
GEOJSON_URL = (
    'https://data.beta.nyc/dataset/3bf5fb73-edb5-4b05-bb29-7c95f4a727fc'
    '/resource/894e9162-871c-4552-a09c-c6915d8783fb'
    '/download/zip_code_040114.geojson'
)
try:
    r = req.get(GEOJSON_URL, timeout=20)
    geojson_data = r.json()
    print(f'✓ GeoJSON loaded: {len(geojson_data["features"])} zip code boundaries')
except Exception as e:
    print(f'✗ GeoJSON fetch failed: {e}')
    geojson_data = None

def get_zip(feature):
    p = feature['properties']
    return str(
        p.get('ZIPCODE') or p.get('zipcode') or
        p.get('ZIP')     or p.get('postalCode', '')
    ).zfill(5)

# ── Build one map given a value extractor and label ───────────────────────────
def build_map(value_key_idx, pollutant_label, health_label_1,
              health_key_idx_1, health_label_2, health_key_idx_2,
              vmin, vmax, circle_scale=0.4):

    m = folium.Map(
        location=[40.7128, -74.006],
        zoom_start=10,
        tiles='CartoDB dark_matter',
        prefer_canvas=True
    )

    colormap = cm.LinearColormap(
        colors=['#2166ac','#4393c3','#74add1','#abd9e9',
                '#ffffbf','#fee090','#fdae61','#f46d43','#d73027','#a50026'],
        vmin=vmin, vmax=vmax,
        caption=f'{pollutant_label} — Blue = Clean | Red = High'
    )
    colormap.add_to(m)

    if geojson_data:
        for feature in geojson_data['features']:
            zc   = get_zip(feature)
            data = zip_dual.get(zc)
            val  = data[value_key_idx] if data else None

            # Zip boundary fill
            folium.GeoJson(
                feature,
                style_function=lambda f, v=val: {
                    'fillColor':   colormap(v) if v else '#2a2a2a',
                    'fillOpacity': 0.72 if v else 0.15,
                    'color':       '#111',
                    'weight':      0.7,
                },
                highlight_function=lambda f: {
                    'fillOpacity': 0.95,
                    'color': '#ffffff',
                    'weight': 2
                },
                tooltip=folium.Tooltip(
                    f"<b>ZIP {zc}</b><br>"
                    f"{data[6] if data else 'N/A'} — {data[7] if data else ''}<br>"
                    f"<b>{pollutant_label}:</b> {val or 'N/A'}<br>"
                    f"<b>{health_label_1}:</b> {data[health_key_idx_1] if data else 'N/A'} per 100k<br>"
                    f"<b>{health_label_2}:</b> {data[health_key_idx_2] if data else 'N/A'} per 100k",
                    sticky=True
                ),
            ).add_to(m)

            # Circle marker sized by primary health outcome
            if data:
                try:
                    centroid = feature['geometry']
                    if centroid['type'] == 'Polygon':
                        coords = centroid['coordinates'][0]
                    else:
                        coords = centroid['coordinates'][0][0]
                    lng = sum(c[0] for c in coords) / len(coords)
                    lat = sum(c[1] for c in coords) / len(coords)
                    radius = data[health_key_idx_1] * circle_scale
                    folium.CircleMarker(
                        location=[lat, lng],
                        radius=max(2, min(radius, 14)),
                        color='white',
                        weight=0.5,
                        fill=True,
                        fill_color=colormap(val),
                        fill_opacity=0.85,
                        tooltip=f"ZIP {zc}: {health_label_1} = {data[health_key_idx_1]} per 100k"
                    ).add_to(m)
                except Exception:
                    pass
    return m

# ── Build both maps ────────────────────────────────────────────────────────────
print('Building Ozone map...')
ozone_map = build_map(
    value_key_idx=0,
    pollutant_label='Ozone AQI',
    health_label_1='Asthma ER Visits (Ozone)',
    health_key_idx_1=1,
    health_label_2='Cardiac & Resp. Deaths (Ozone)',
    health_key_idx_2=2,
    vmin=35, vmax=90,
    circle_scale=0.08
)

print('Building PM2.5 map...')
pm25_map = build_map(
    value_key_idx=3,
    pollutant_label='PM2.5 (µg/m³)',
    health_label_1='Cardio Hospitalizations (PM2.5)',
    health_key_idx_1=4,
    health_label_2='Respiratory Hospitalizations (PM2.5)',
    health_key_idx_2=5,
    vmin=6, vmax=15,
    circle_scale=0.18
)

# Save both
ozone_path = os.path.join('..', 'Notebooks', 'ozone_map.html')
pm25_path  = os.path.join('..', 'Notebooks', 'pm25_map.html')
ozone_map.save(ozone_path)
pm25_map.save(pm25_path)
print(f'✓ Saved ozone_map.html')
print(f'✓ Saved pm25_map.html')

In [ ]:
# ── Display side by side in notebook ──────────────────────────────────────────
ozone_html = ozone_map._repr_html_()
pm25_html  = pm25_map._repr_html_()

side_by_side = f"""
<div style='background:#0f1117; padding:12px; border-radius:10px;'>
  <div style='display:flex; gap:12px; align-items:flex-start;'>
    <div style='flex:1;'>
      <div style='color:white; font-family:Segoe UI; font-weight:bold;
                  font-size:14px; margin-bottom:6px; text-align:center;
                  background:#1a1d27; padding:8px; border-radius:6px;'>
        🌫 Ozone → Asthma ER + Cardiac Deaths
      </div>
      <div style='height:480px; border-radius:8px; overflow:hidden;
                  border:1px solid #2a2d3a;'>
        {ozone_html}
      </div>
    </div>
    <div style='flex:1;'>
      <div style='color:white; font-family:Segoe UI; font-weight:bold;
                  font-size:14px; margin-bottom:6px; text-align:center;
                  background:#1a1d27; padding:8px; border-radius:6px;'>
        🏭 PM2.5 → Cardio + Respiratory Hospitalizations
      </div>
      <div style='height:480px; border-radius:8px; overflow:hidden;
                  border:1px solid #2a2d3a;'>
        {pm25_html}
      </div>
    </div>
  </div>
  <div style='color:#555; font-family:Segoe UI; font-size:11px;
              text-align:center; margin-top:10px;'>
    Circle size = hospitalization rate &nbsp;|&nbsp;
    Blue = lower pollution/disease &nbsp;|&nbsp;
    Red = higher pollution/disease &nbsp;|&nbsp;
    Hover any zip code for full data
  </div>
</div>
"""

display(HTML(side_by_side))
print('✓ Side-by-side heatmaps displayed')